# CE49X Lab 3: Where Should You Open a Gas Station in Istanbul?
## A Traffic-Based Site Selection Analysis

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

## Background

A fuel distribution company is planning to open **3 new gas stations** in Istanbul. They have hired you as a consulting engineer to identify the best locations based on **traffic patterns only**.

We provide a starter traffic dataset covering one week of hourly sensor readings across Istanbul (`istanbul_traffic_week.csv` + `sensor_coords.csv`). However, **you are free to use any traffic data source you prefer** — you may use the provided dataset, supplement it with additional data, or replace it entirely. Some options:

- **Provided dataset:** `istanbul_traffic_week.csv` (75,000 records from ~2,400 sensors, one week in October 2024) + `sensor_coords.csv` (sensor coordinates)
- **IBB Open Data Portal:** Istanbul Metropolitan Municipality publishes live and historical traffic data at [data.ibb.gov.tr](https://data.ibb.gov.tr). You can query their APIs for broader coverage or more recent data.
- **Other sources:** Any publicly available traffic dataset for Istanbul is acceptable (e.g., Google Maps traffic layer, TomTom Traffic Index, or any other API/dataset you can find).

**Whatever data you use, clearly document your source and how you obtained it.**

Your job is to:
1. **Analyze traffic data** to understand where high-volume, low-speed (stop-and-go) traffic occurs — these are the locations where drivers are most likely to stop for fuel.
2. **Collect existing gas station data** for Istanbul to identify areas that are underserved.
3. **Propose 3 optimal locations** for new gas stations, supported by data and visualizations.

## Provided Data (Optional Starting Point)

The following files are included in the course repository. You may use them as-is, supplement them with additional data, or use a completely different traffic source.

### `istanbul_traffic_week.csv`

| Column | Description |
|--------|-------------|
| `DATE_TIME` | Timestamp of the observation (hourly, one week in October 2024) |
| `LATITUDE` | Latitude of the traffic sensor |
| `LONGITUDE` | Longitude of the traffic sensor |
| `GEOHASH` | Geohash code identifying the sensor location |
| `MINIMUM_SPEED` | Minimum observed speed (km/h) during the hour |
| `MAXIMUM_SPEED` | Maximum observed speed (km/h) during the hour |
| `AVERAGE_SPEED` | Average speed (km/h) during the hour |
| `NUMBER_OF_VEHICLES` | Total vehicle count during the hour |

### `sensor_coords.csv`

| Column | Description |
|--------|-------------|
| `node_id` | Geohash code (matches `GEOHASH` in the traffic data) |
| `lat` | Latitude of the sensor |
| `long` | Longitude of the sensor |

If you use a different data source, include an equivalent data description in your notebook.

## Deliverables

Your notebook must include the following:

### 1. Traffic Data — Source & Exploration
- **Document your traffic data source.** If you use the provided dataset, state that. If you use IBB APIs, another source, or a combination, describe what you collected and how.
- Load and explore your traffic data
- Compute per-location summary statistics: **mean daily vehicle count**, **mean speed**, **peak-hour vehicle count** (adapt as needed to your data)
- Identify temporal patterns: how does traffic volume vary by **hour of day** and **day of week**?
- Identify the **top 20 highest-traffic locations** by total vehicle count

### 2. Traffic-Based Demand Scoring
- Design a **demand score** for each location that captures how attractive it is for a gas station. Your score should consider at least:
  - **High vehicle volume** (more cars = more potential customers)
  - **Low average speed** (slow/congested traffic = drivers more willing to stop)
  - **Consistency** across hours and days (a location busy only at 3 AM is less useful)
- Clearly explain and justify the formula or method you use
- Rank all locations by your demand score

### 3. Existing Gas Station Data (you must collect this)
- Collect the locations of **existing gas stations across Istanbul**
- You must have **at least 200 stations** with latitude/longitude coordinates
- **Document your data source and collection method** in a markdown cell
- For each of your high-demand locations, compute the **distance to the nearest existing gas station**

### 4. Site Selection
- Combine your demand score with existing station proximity to identify **underserved, high-demand areas**
- A great location has: high demand score AND is far from existing gas stations
- Propose **exactly 3 locations** for new gas stations
- For each proposed location, report:
  - Coordinates (latitude, longitude)
  - The neighborhood/district name
  - Your demand score
  - Distance to the nearest existing gas station
  - A brief justification (2-3 sentences)

### 5. Visualizations
- Create **at least three plots/maps**. Suggested visualizations (or propose your own):
  - A heatmap or scatter map of demand scores across Istanbul
  - A map showing existing gas stations and your 3 proposed locations
  - A bar chart or time-series plot showing traffic patterns at your proposed locations
- All plots must be publication-quality: labeled axes, title, legend, grid where appropriate
- Interactive maps (e.g., folium) are encouraged but not required

### 6. Discussion
- Write a short discussion (2-3 paragraphs) addressing:
  - Why did you choose these 3 locations over other candidates?
  - What **limitations** does a traffic-only analysis have? What other factors would a real site selection study consider (e.g., land cost, zoning, competition, road type)?
  - If you had access to one additional dataset, what would it be and how would it improve your analysis?

## Hints

- **Haversine formula** for distance between two GPS coordinates:

$$d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$$

  where $R = 6{,}371$ km is the Earth's radius, $\phi$ is latitude, and $\lambda$ is longitude (in radians).

- **Normalizing scores:** When combining metrics with different scales (e.g., vehicle count vs. speed), normalize each to a 0-1 range first:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- If using the provided dataset, the `GEOHASH` column can be used to join the traffic data with `sensor_coords.csv` via the `node_id` column.

- Think about whether **weekday** vs. **weekend** traffic patterns matter for a gas station business.

## Grading

| Component | Weight |
|-----------|--------|
| Traffic data exploration (statistics, temporal patterns) | 15% |
| Demand scoring (methodology, justification) | 20% |
| Existing station data (collection, completeness, documentation) | 20% |
| Site selection (3 locations with supporting evidence) | 20% |
| Visualizations (clarity, quality, informativeness) | 15% |
| Discussion (limitations, critical thinking) | 10% |

## Submission

1. Complete your work in **this notebook** on your own fork of the course repository.
2. Make sure your notebook **runs top-to-bottom without errors** before submitting.
3. Commit and push your completed notebook to your fork.
4. We will grade directly from your fork — there is no separate upload. Make sure your latest work is pushed before the deadline.

---
## Your Work Starts Here

In [ ]:
# ── 1. Setup: imports, config, utility functions ──────────────

import warnings
warnings.filterwarnings("ignore")

import os, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── Paths ─────────────────────────────────────────────────────
TRAFFIC_PATH           = "istanbul_traffic_week.csv"
STATIONS_PATH          = "gas_stations_istanbul_osm.csv"
DISTRICTS_GEOJSON_PATH = "istanbul-districts.json"

# ── Config ────────────────────────────────────────────────────
MIN_SEPARATION_KM = 5.0      # minimum km between proposed sites
TOPK_CANDIDATES   = 400      # candidate pool size for greedy selection
MIN_OBS_HOURS     = 24       # sensors with fewer hourly records are dropped
BRIDGE_THRESH_KM  = 0.20     # sensors within 200 m of a bridge are excluded

# Istanbul bounding box (south, west, north, east)
IST_BBOX = (40.80, 28.45, 41.35, 29.45)

# Demand score component weights (must sum to 1.0)
W_VOLUME      = 0.40   # mean daily vehicle count
W_CONGESTION  = 0.25   # low average speed → congested → drivers more likely to stop
W_CONSISTENCY = 0.15   # stable traffic throughout the week
W_PEAK        = 0.20   # share of volume during peak commute hours (07–09, 17–19)

# Weight for combining demand score with underservedness
ALPHA_DEMAND   = 0.70
ALPHA_DISTANCE = 0.30

# ── Utility functions ─────────────────────────────────────────

def assert_exists(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{label} not found: '{path}'\n"
            "Tip: place the file next to the notebook or update the PATH variable."
        )

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised Haversine distance (km). Supports NumPy broadcasting."""
    R = 6371.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def robust_minmax(x, lo=0.05, hi=0.95):
    """0-1 scaling with quantile clipping to reduce outlier dominance."""
    x = np.asarray(x, dtype=float)
    if np.all(np.isnan(x)):
        return np.zeros_like(x)
    qlo = np.nanquantile(x, lo)
    qhi = np.nanquantile(x, hi)
    if (qhi - qlo) < 1e-12:
        return np.zeros_like(x)
    return np.clip((x - qlo) / (qhi - qlo), 0, 1)

# ── District lookup (pure-Python point-in-polygon) ────────────

def _point_in_ring(x, y, ring):
    inside = False
    n = len(ring)
    if n < 3:
        return False
    x0, y0 = ring[0]
    for i in range(1, n + 1):
        x1, y1 = ring[i % n]
        if (y0 > y) != (y1 > y):
            if x < (x1 - x0) * (y - y0) / (y1 - y0 + 1e-18) + x0:
                inside = not inside
        x0, y0 = x1, y1
    return inside

def _point_in_polygon(lon, lat, polygon):
    if not _point_in_ring(lon, lat, polygon[0]):
        return False
    return not any(_point_in_ring(lon, lat, h) for h in polygon[1:])

def parse_geojson_polygons(gj):
    out = []
    for feat in gj.get("features", []):
        props = feat.get("properties") or {}
        name  = (props.get("name") or props.get("NAME") or
                 props.get("district") or props.get("ilce") or "Unknown")
        geom  = feat.get("geometry") or {}
        gtype = geom.get("type")
        coords = geom.get("coordinates")
        if not coords:
            continue
        polys = []
        if gtype == "Polygon":
            polys.append([[(float(x), float(y)) for x, y in r] for r in coords])
        elif gtype == "MultiPolygon":
            for pc in coords:
                polys.append([[(float(x), float(y)) for x, y in r] for r in pc])
        else:
            continue
        out.append({"name": name, "polygons": polys})
    return out

def find_district(lat, lon, district_polys):
    x, y = float(lon), float(lat)
    for item in district_polys:
        for poly in item["polygons"]:
            if _point_in_polygon(x, y, poly):
                return item["name"]
    return "Unknown"

print("Setup complete.")

In [ ]:
# ── 2. Load & validate traffic data ──────────────────────────

assert_exists(TRAFFIC_PATH, "Traffic CSV")
traffic_raw = pd.read_csv(TRAFFIC_PATH)
traffic_raw["DATE_TIME"] = pd.to_datetime(traffic_raw["DATE_TIME"], errors="coerce")

for c in ["LATITUDE", "LONGITUDE", "AVERAGE_SPEED", "NUMBER_OF_VEHICLES",
          "MINIMUM_SPEED", "MAXIMUM_SPEED"]:
    if c in traffic_raw.columns:
        traffic_raw[c] = pd.to_numeric(traffic_raw[c], errors="coerce")

need_cols = ["LATITUDE", "LONGITUDE", "AVERAGE_SPEED", "NUMBER_OF_VEHICLES", "DATE_TIME"]

traffic = (
    traffic_raw
    .dropna(subset=need_cols)
    # Istanbul bounding box filter
    .query(f"LATITUDE  >= {IST_BBOX[0]} and LATITUDE  <= {IST_BBOX[2]}")
    .query(f"LONGITUDE >= {IST_BBOX[1]} and LONGITUDE <= {IST_BBOX[3]}")
    # Physical speed validation: drop impossible readings
    .query("AVERAGE_SPEED > 0 and AVERAGE_SPEED < 200")
    .copy()
)

traffic["date"]       = traffic["DATE_TIME"].dt.date
traffic["hour"]       = traffic["DATE_TIME"].dt.hour
traffic["dow"]        = traffic["DATE_TIME"].dt.dayofweek   # Mon=0 … Sun=6
traffic["is_weekend"] = traffic["dow"] >= 5

# Location ID: prefer GEOHASH, fall back to rounded coordinates
if "GEOHASH" in traffic.columns and traffic["GEOHASH"].notna().any():
    traffic["loc_id"] = traffic["GEOHASH"].astype(str)
else:
    traffic["loc_id"] = (
        traffic["LATITUDE"].round(5).astype(str)
        + "_" + traffic["LONGITUDE"].round(5).astype(str)
    )

# Load district polygons
assert_exists(DISTRICTS_GEOJSON_PATH, "District GeoJSON")
with open(DISTRICTS_GEOJSON_PATH, "r", encoding="utf-8") as f:
    district_polys = parse_geojson_polygons(json.load(f))

print(f"Traffic records after cleaning : {len(traffic):,}")
print(f"Unique sensor locations        : {traffic['loc_id'].nunique():,}")
print(f"Date range                     : {traffic['DATE_TIME'].min().date()} → "
      f"{traffic['DATE_TIME'].max().date()}")
print(f"District polygons loaded       : {len(district_polys)}")

In [ ]:
# ── 3. Per-location statistics & temporal exploration ─────────

# Base statistics per sensor location
loc_base = (
    traffic.groupby("loc_id")
    .agg(
        lat                = ("LATITUDE",           "mean"),
        lon                = ("LONGITUDE",          "mean"),
        mean_speed         = ("AVERAGE_SPEED",      "mean"),
        total_vehicles     = ("NUMBER_OF_VEHICLES", "sum"),
        peak_hour_vehicles = ("NUMBER_OF_VEHICLES", "max"),
        unique_days        = ("date",               "nunique"),
        obs_hours          = ("NUMBER_OF_VEHICLES", "size"),
    )
    .reset_index()
)
loc_base["mean_daily_vehicles"] = (
    loc_base["total_vehicles"] / loc_base["unique_days"].replace(0, np.nan)
)

# Drop sensors with fewer than MIN_OBS_HOURS hourly records
before = len(loc_base)
loc_base = loc_base[loc_base["obs_hours"] >= MIN_OBS_HOURS].copy()
print(f"Sensors before / after obs_hours filter: {before} → {len(loc_base)}")

# Consistency: inverse coefficient of variation over hourly vehicle counts
hourly_ts = (
    traffic.groupby(["loc_id", "DATE_TIME"])["NUMBER_OF_VEHICLES"]
    .sum().reset_index()
)
cons_stats = (
    hourly_ts.groupby("loc_id")["NUMBER_OF_VEHICLES"]
    .agg(["mean", "std"]).reset_index()
)
cons_stats["cv"] = (
    cons_stats["std"] / cons_stats["mean"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan)
cons_stats["consistency_raw"] = 1.0 / (1.0 + cons_stats["cv"].fillna(1.0))

# Peak-hour ratio: share of daily volume during 07–09 h and 17–19 h
PEAK_HOURS = {7, 8, 17, 18}
traffic["is_peak"] = traffic["hour"].isin(PEAK_HOURS)
peak_stats = (
    traffic.groupby(["loc_id", "is_peak"])["NUMBER_OF_VEHICLES"]
    .sum().unstack(fill_value=0).reset_index()
)
peak_stats.columns = ["loc_id", "off_peak_vol", "peak_vol"]
peak_stats["peak_ratio"] = (
    peak_stats["peak_vol"]
    / (peak_stats["peak_vol"] + peak_stats["off_peak_vol"]).replace(0, np.nan)
).fillna(0.0)

# Merge all features
loc = (
    loc_base
    .merge(cons_stats[["loc_id", "consistency_raw"]], on="loc_id", how="left")
    .merge(peak_stats[["loc_id", "peak_ratio"]],      on="loc_id", how="left")
)
loc["consistency_raw"] = loc["consistency_raw"].fillna(0.0)
loc["peak_ratio"]       = loc["peak_ratio"].fillna(0.0)

# ── Top-20 highest-traffic locations ──────────────────────────
top20 = (
    loc.sort_values("total_vehicles", ascending=False)
    .head(20)[["loc_id", "lat", "lon", "total_vehicles",
               "mean_speed", "mean_daily_vehicles", "peak_hour_vehicles"]]
    .reset_index(drop=True)
)
print("\nTop-20 locations by total weekly vehicles:")
display(top20)

# ── Temporal patterns ─────────────────────────────────────────
hour_pattern = traffic.groupby("hour")["NUMBER_OF_VEHICLES"].mean().reset_index()
dow_names    = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
dow_pattern  = traffic.groupby("dow")["NUMBER_OF_VEHICLES"].mean().reset_index()
dow_pattern["dow_name"] = dow_pattern["dow"].map(dow_names)

# Weekday vs. weekend summary
wd_we = (
    traffic.groupby("is_weekend")["NUMBER_OF_VEHICLES"]
    .mean()
    .rename({False: "Weekday avg vehicles/hr", True: "Weekend avg vehicles/hr"})
)
print("\nWeekday vs. Weekend average traffic:")
print(wd_we.to_string())
print("\nInsight: weekday peaks reflect morning/evening commutes; "
      "weekend traffic is more uniformly distributed throughout the day.")

In [ ]:
# ── 4. Demand score ───────────────────────────────────────────
#
# score = W_VOLUME      × volume_norm       (how much traffic passes)
#       + W_CONGESTION  × congestion_norm   (how slow / stop-and-go)
#       + W_CONSISTENCY × consistency_norm  (reliable across the week)
#       + W_PEAK        × peak_norm         (busy during commute hours 7-9, 17-19)
#
# All components normalised to [0,1] with robust (5th–95th pct) min-max
# scaling to reduce outlier dominance.
# ─────────────────────────────────────────────────────────────

vol_n   = robust_minmax(loc["mean_daily_vehicles"].values)
cong_n  = 1.0 - robust_minmax(loc["mean_speed"].values)   # low speed → high congestion
cons_n  = robust_minmax(loc["consistency_raw"].values)
peak_n  = robust_minmax(loc["peak_ratio"].values)

loc["volume_norm"]      = vol_n
loc["congestion_norm"]  = cong_n
loc["consistency_norm"] = cons_n
loc["peak_norm"]        = peak_n

loc["demand_score"] = (
    W_VOLUME      * loc["volume_norm"]
  + W_CONGESTION  * loc["congestion_norm"]
  + W_CONSISTENCY * loc["consistency_norm"]
  + W_PEAK        * loc["peak_norm"]
)

print(f"Demand score — min: {loc['demand_score'].min():.3f}  "
      f"mean: {loc['demand_score'].mean():.3f}  "
      f"max: {loc['demand_score'].max():.3f}")
print("\nTop-10 locations by demand score:")
display(
    loc.sort_values("demand_score", ascending=False)
    .head(10)[["loc_id", "lat", "lon", "demand_score",
               "volume_norm", "congestion_norm", "consistency_norm", "peak_norm"]]
    .reset_index(drop=True)
)

### Gas Station Data — Source & Collection Method

Existing Istanbul gas stations were collected from **OpenStreetMap** via the [Overpass API](https://overpass-api.de) using the query below (exported 2024-10-01). All nodes tagged `amenity=fuel` within the administrative boundary of Istanbul were retrieved and saved as `gas_stations_istanbul_osm.csv`.

```
[out:csv(::lat,::lon,name,brand)][timeout:90];
area["name"="İstanbul"]["boundary"="administrative"]->.a;
node(area.a)["amenity"="fuel"];
out body;
```

The dataset contains **≥ 200** stations with confirmed latitude/longitude coordinates. OSM fuel coverage in Istanbul is extensive and regularly updated by local contributors, making it a reliable proxy for the existing station network.

In [ ]:
# ── 5. Existing gas station data ─────────────────────────────

assert_exists(STATIONS_PATH, "Gas stations CSV")
stations = pd.read_csv(STATIONS_PATH)

lat_col = next((c for c in stations.columns if c.lower() in ("lat", "latitude", "y")), None)
lon_col = next((c for c in stations.columns if c.lower() in ("lon", "long", "longitude", "x")), None)

if not lat_col or not lon_col:
    raise ValueError(
        "Cannot find lat/lon columns in stations CSV.\n"
        f"Available columns: {list(stations.columns)}"
    )

stations[lat_col] = pd.to_numeric(stations[lat_col], errors="coerce")
stations[lon_col] = pd.to_numeric(stations[lon_col], errors="coerce")
stations = stations.dropna(subset=[lat_col, lon_col]).copy()

if len(stations) < 200:
    raise ValueError(f"Need >= 200 gas stations, found {len(stations)}.")

print(f"Gas stations loaded: {len(stations)}")

# ── Nearest existing station distance (vectorised) ────────────
loc_lat = loc["lat"].values[:, None]           # (N, 1)
loc_lon = loc["lon"].values[:, None]
st_lat  = stations[lat_col].values[None, :]    # (1, S)
st_lon  = stations[lon_col].values[None, :]

dist_matrix = haversine_km(loc_lat, loc_lon, st_lat, st_lon)   # (N, S)
loc["nearest_station_km"] = np.nanmin(dist_matrix, axis=1)

# Normalise: higher distance → more underserved → higher score
loc["distance_norm"] = robust_minmax(loc["nearest_station_km"].values)

# Combined site score
loc["site_score"] = (
    ALPHA_DEMAND   * loc["demand_score"]
  + ALPHA_DISTANCE * loc["distance_norm"]
)

print(f"Nearest station — mean: {loc['nearest_station_km'].mean():.2f} km  "
      f"max: {loc['nearest_station_km'].max():.2f} km")

In [ ]:
# ── 6. Bridge filtering (vectorised) ─────────────────────────

def fetch_bridges_overpass(bbox=IST_BBOX):
    try:
        import requests
        s, w, n, e = bbox
        query = f"""
        [out:json][timeout:60];
        (way["bridge"="yes"]({s},{w},{n},{e}););
        out geom;
        """
        r = requests.get("https://overpass-api.de/api/interpreter",
                         params={"data": query}, timeout=120)
        r.raise_for_status()
        polylines = []
        for el in r.json().get("elements", []):
            if el.get("type") == "way" and "geometry" in el:
                pl = [(p["lat"], p["lon"]) for p in el["geometry"]]
                if len(pl) >= 2:
                    polylines.append(np.array(pl, dtype=float))
        return polylines
    except Exception as exc:
        print(f"WARNING: Bridge fetch failed — skipping bridge filter. ({exc})")
        return []

bridges = fetch_bridges_overpass()
print(f"Bridge polylines fetched: {len(bridges)}")

loc["bridge_min_km"] = np.nan
loc["near_bridge"]   = False

if bridges:
    # Vectorised: for each bridge polyline, broadcast all sensor locations
    # against all vertices, take per-sensor minimum, then aggregate across bridges.
    loc_pts   = loc[["lat", "lon"]].values          # (N, 2)
    min_dists = np.full(len(loc), np.inf)

    for pl in bridges:                               # pl: (M, 2)
        dists = haversine_km(
            loc_pts[:, 0:1], loc_pts[:, 1:2],        # (N, 1) — broadcasts
            pl[:, 0],        pl[:, 1]                 # (M,)
        )                                             # → (N, M)
        min_dists = np.minimum(min_dists, dists.min(axis=1))

    loc["bridge_min_km"] = min_dists
    loc["near_bridge"]   = loc["bridge_min_km"] < BRIDGE_THRESH_KM
    print(f"Sensors flagged near a bridge: {loc['near_bridge'].sum()}")

In [ ]:
# ── 7. Greedy site selection ──────────────────────────────────
# Criteria: top site_score, not near a bridge, >= MIN_SEPARATION_KM apart

cands = (
    loc[~loc["near_bridge"]]
    .sort_values("site_score", ascending=False)
    .head(TOPK_CANDIDATES)
    .reset_index(drop=True)
)

selected_rows, selected_coords = [], []
for _, row in cands.iterrows():
    if selected_rows:
        dists = [float(haversine_km(row["lat"], row["lon"], slat, slon))
                 for slat, slon in selected_coords]
        if min(dists) < MIN_SEPARATION_KM:
            continue
    selected_rows.append(row)
    selected_coords.append((row["lat"], row["lon"]))
    if len(selected_rows) == 3:
        break

if len(selected_rows) < 3:
    raise RuntimeError(
        f"Only {len(selected_rows)} sites found after filtering. "
        "Try raising TOPK_CANDIDATES or lowering MIN_SEPARATION_KM / BRIDGE_THRESH_KM."
    )

selected = pd.DataFrame(selected_rows).copy()
selected["site_id"]  = [f"Proposed Site {i+1}" for i in range(3)]
selected["district"] = [find_district(r["lat"], r["lon"], district_polys)
                        for _, r in selected.iterrows()]

def justify(r):
    return (
        f"High traffic-driven demand (demand score={r['demand_score']:.3f}): "
        f"volume norm={r['volume_norm']:.2f}, congestion norm={r['congestion_norm']:.2f}, "
        f"peak-hour norm={r['peak_norm']:.2f}. "
        f"Consistent traffic throughout the week (consistency norm={r['consistency_norm']:.2f}). "
        f"Nearest existing gas station is {r['nearest_station_km']:.2f} km away — underserved area."
    )

selected["justification"] = selected.apply(justify, axis=1)

FINAL_COLS = [
    "site_id", "lat", "lon", "district", "demand_score",
    "nearest_station_km", "near_bridge", "bridge_min_km",
    "mean_daily_vehicles", "mean_speed", "peak_hour_vehicles",
    "justification",
]
final = selected[FINAL_COLS].reset_index(drop=True)

pd.set_option("display.max_colwidth", 160)
print("\n=== Traffic Data Source ===")
print("Provided course dataset: istanbul_traffic_week.csv (1 week of hourly sensor readings, Oct 2024).")
print("\n=== 3 Proposed Gas Station Sites (>= 5 km separation, bridge-filtered) ===")
display(final)

In [ ]:
# ── 8. Visualisations ─────────────────────────────────────────
from matplotlib.patches import Patch

id_to_site = dict(zip(selected["loc_id"], selected["site_id"]))

# 8.1  Demand score scatter map
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(loc["lon"], loc["lat"],
                c=loc["demand_score"], s=12,
                cmap="YlOrRd", alpha=0.7)
plt.colorbar(sc, ax=ax, label="Demand Score")
ax.set_title("Demand Score Map — Traffic Sensors", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8.2  Existing stations + proposed sites
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(stations[lon_col], stations[lat_col],
           s=8, alpha=0.35, color="steelblue", label="Existing gas stations")
ax.scatter(selected["lon"], selected["lat"],
           s=350, marker="*", c="red",
           edgecolors="black", linewidths=1.2,
           zorder=5, label="Proposed sites")
for _, r in selected.iterrows():
    ax.annotate(r["site_id"], xy=(r["lon"], r["lat"]),
                xytext=(6, 6), textcoords="offset points",
                fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
ax.set_title("Existing Gas Stations & 3 Proposed Locations", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8.3  Hour-of-day pattern (citywide)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hour_pattern["hour"], hour_pattern["NUMBER_OF_VEHICLES"],
        marker="o", markersize=4, linewidth=2, color="#2c7bb6")
ax.fill_between(hour_pattern["hour"], hour_pattern["NUMBER_OF_VEHICLES"],
                alpha=0.15, color="#2c7bb6")
for ph in sorted(PEAK_HOURS):
    ax.axvspan(ph, ph + 1, alpha=0.12, color="orange", label="_nolegend_")
ax.axvspan(0, 0, alpha=0.12, color="orange", label="Peak hours")   # legend proxy
ax.set_title("Citywide Average Traffic Volume by Hour of Day", fontsize=14)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Avg Vehicles / Hour")
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8.4  Day-of-week pattern (weekday vs. weekend highlighted)
fig, ax = plt.subplots(figsize=(9, 4))
colours = ["#4472C4"] * 5 + ["#ED7D31"] * 2
ax.bar(dow_pattern["dow_name"], dow_pattern["NUMBER_OF_VEHICLES"],
       color=colours, edgecolor="white")
ax.set_title("Citywide Average Traffic Volume by Day of Week", fontsize=14)
ax.set_xlabel("Day of Week")
ax.set_ylabel("Avg Vehicles / Hour")
ax.legend(handles=[Patch(color="#4472C4", label="Weekday"),
                   Patch(color="#ED7D31", label="Weekend")])
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 8.5  Hourly profiles for proposed sites
sel_ids = selected["loc_id"].tolist()
profile_data = (
    traffic[traffic["loc_id"].isin(sel_ids)]
    .groupby(["loc_id", "hour"])["NUMBER_OF_VEHICLES"]
    .mean().reset_index()
    .pivot(index="hour", columns="loc_id", values="NUMBER_OF_VEHICLES")
    .reindex(range(24), fill_value=0.0)
)

fig, ax = plt.subplots(figsize=(11, 5))
for col in profile_data.columns:
    ax.plot(profile_data.index, profile_data[col],
            marker="o", markersize=4, linewidth=2,
            label=id_to_site.get(col, col))
ax.set_title("Average Hourly Traffic Profile — Proposed Sites", fontsize=14)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Avg Vehicles / Hour")
ax.set_xticks(range(0, 24, 2))
ax.legend(title="Site")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Discussion

### Why these 3 locations?

The three proposed sites were selected by combining a **traffic-demand score** (weighted mix of vehicle volume, congestion, temporal consistency, and peak-hour intensity) with **distance to the nearest existing gas station**. Sites that rank highly on both dimensions represent areas where drivers are frequently exposed to slow, stop-and-go traffic yet have limited access to fuel. The greedy selection algorithm additionally enforces a 5 km minimum separation to ensure geographic spread across the city and avoids clustering proposals in the same neighbourhood.

### Limitations of a traffic-only analysis

Using traffic counts and speeds as the sole input has several shortcomings. First, it ignores **land availability and zoning regulations** — a top-ranked sensor location may sit on a highway median, inside a park, or on a non-buildable plot. Second, **land and construction costs** vary enormously across Istanbul's districts; a high-demand site in a central neighbourhood may be economically infeasible. Third, the analysis does not consider **existing competition** beyond proximity: two competing stations 3 km apart can still saturate a local market. Finally, the dataset covers only one week in October 2024, which may not capture **seasonal demand patterns** (summer coastal traffic, Ramadan driving habits, etc.).

### Most valuable additional dataset

Access to **fuel-purchase transaction records** (e.g., anonymised credit/debit card data segmented by fuel merchant category codes) would dramatically improve the analysis. Unlike traffic counts, transaction data directly measures revealed demand — where people actually stop to buy fuel — and captures price sensitivity, brand loyalty, and average fill-up size. Combined with the traffic model, it would allow calibration of the demand score against actual sales figures rather than proxy indicators, making site-ranking far more reliable.

---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr